In [ ]:
import os, sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "--quiet"], check=True)

from dotenv import load_dotenv, find_dotenv

dotenv_path = ""
for candidate in [
    "/Users/jacindabietz/corn_nb/.env",
    os.path.join(os.getcwd(), ".env"),
    os.path.expanduser("~/.env"),
]:
    if os.path.exists(candidate):
        dotenv_path = candidate
        break
if not dotenv_path:
    dotenv_path = find_dotenv(usecwd=True) or ""

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Loaded .env from: {dotenv_path}")
    print("NASS_API_KEY present:", bool(os.getenv("NASS_API_KEY")))
    print("USDA_NASS_API_KEY present:", bool(os.getenv("USDA_NASS_API_KEY")))
    print("NASS_KEY present:", bool(os.getenv("NASS_KEY")))
else:
    print("WARNING: No .env file found")

ANTHROPIC_API_KEY  = os.getenv("ANTHROPIC_API_KEY", "")
SH_CLIENT_ID       = os.getenv("SH_CLIENT_ID", "")
SH_CLIENT_SECRET   = os.getenv("SH_CLIENT_SECRET", "")
NASS_API_KEY       = os.getenv("NASS_API_KEY", "")
EARTHDATA_USERNAME = os.getenv("EARTHDATA_USERNAME", "")
EARTHDATA_PASSWORD = os.getenv("EARTHDATA_PASSWORD", "")

required = [
    "ANTHROPIC_API_KEY", "SH_CLIENT_ID", "SH_CLIENT_SECRET",
    "NASS_API_KEY", "EARTHDATA_USERNAME", "EARTHDATA_PASSWORD",
]
missing = [name for name in required if not os.getenv(name)]
if missing:
    print("Missing env vars:", ", ".join(missing))
else:
    print("All required env vars are loaded.")


In [ ]:
# Cell 2 - Dependencies
%pip install \
    numpy>=2.3.3 \
    rasterio geopandas shapely pyproj fiona \
    sentinelhub \
    planetary-computer pystac-client odc-stac \
    earthaccess \
    anthropic \
    xgboost lightgbm shap \
    timm transformers huggingface_hub \
    quantile-forest \
    folium matplotlib seaborn \
    requests pandas numpy scikit-learn \
    scikit-image \
    --quiet

print("Dependencies installed.")

In [ ]:
# Cell 2b -- Optional compatibility pin and kernel restart
import platform
import subprocess
import sys

# Keep NumPy aligned with the installed geospatial/ML stack on Python 3.14
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy>=2.3.3", "--quiet"], check=True)

# Upgrading torch/torchvision is optional and often unnecessary on macOS
if platform.system() != "Darwin":
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "torch", "torchvision", "--quiet"], check=False)
else:
    print("Skipping torch/torchvision upgrade on macOS.")

import importlib.metadata as metadata
print(f"NumPy version on disk: {metadata.version('numpy')}")
print("Restart the kernel once after this cell so imported packages see the updated NumPy.")

In [ ]:
# Cell 3 - Directory setup
import os

dirs = [
    "/Users/jacindabietz/corn_nb/data/nass",
    "/Users/jacindabietz/corn_nb/data/sentinel",
    "/Users/jacindabietz/corn_nb/data/hls",
    "/Users/jacindabietz/corn_nb/data/cdl",
    "/Users/jacindabietz/corn_nb/data/naip",
    "/Users/jacindabietz/corn_nb/data/weather",
    "/Users/jacindabietz/corn_nb/data/embeddings",
    "/Users/jacindabietz/corn_nb/data/naip_features",
    "/Users/jacindabietz/corn_nb/data/features",
    "/Users/jacindabietz/corn_nb/models",
    "/Users/jacindabietz/corn_nb/outputs",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Directories ready:")
for d in dirs:
    print(f"  {d}")

In [ ]:
# Cell 4 - NASS QuickStats: corn yield data 2005-2024 (state + county)
import os, requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

STATE_CSV  = '/Users/jacindabietz/corn_nb/data/nass/corn_yields_state.csv'
COUNTY_CSV = '/Users/jacindabietz/corn_nb/data/nass/corn_yields_county.csv'

NASS_API_KEY = (
    os.getenv('NASS_API_KEY')
    or os.getenv('USDA_NASS_API_KEY')
    or os.getenv('NASS_KEY')
    or ''
)
NASS_URL     = 'https://quickstats.nass.usda.gov/api/api_GET/'
STATES       = ['IOWA', 'COLORADO', 'WISCONSIN', 'MISSOURI', 'NEBRASKA']

COMMON_PARAMS = dict(
    key               = NASS_API_KEY,
    source_desc       = 'SURVEY',
    commodity_desc    = 'CORN',
    statisticcat_desc = 'YIELD',
    unit_desc         = 'BU / ACRE',
    freq_desc         = 'ANNUAL',
    year__GE          = '2005',
    year__LE          = '2024',
    format            = 'JSON',
)


def _fetch_nass(agg_level):
    if not NASS_API_KEY or NASS_API_KEY.strip().lower() in {'', 'changeme', 'your_key_here'}:
        raise RuntimeError('NASS_API_KEY is missing or looks like a placeholder. Add a valid USDA QuickStats API key to your .env and rerun this cell.')
    params = {**COMMON_PARAMS, 'agg_level_desc': agg_level}
    r = requests.get(NASS_URL, params=params, timeout=60)
    if r.status_code == 401:
        raise RuntimeError(
            'USDA NASS QuickStats returned 401 Unauthorized. ',
            'Check that NASS_API_KEY is valid and not expired.'
        )
    r.raise_for_status()
    data = r.json().get('data', [])
    if not data:
        raise RuntimeError(f'NASS returned no records for agg_level={agg_level}')
    df = pd.DataFrame(data)
    # strip commas from Value, drop suppressed rows
    df = df[df['Value'].str.strip() != '(D)'].copy()
    df['yield_bu_acre'] = df['Value'].str.replace(',', '', regex=False).astype(float)
    df['year']          = df['year'].astype(int)
    df['state_name']    = df['state_name'].str.title()
    return df


if os.path.exists(STATE_CSV) and os.path.exists(COUNTY_CSV):
    state_df  = pd.read_csv(STATE_CSV)
    county_df = pd.read_csv(COUNTY_CSV)
    print('Loaded cached NASS CSVs.')
else:
    try:
        print('Fetching NASS state-level data ...')
        raw_state = _fetch_nass('STATE')
        state_df  = (raw_state[raw_state['state_name'].str.upper()
                                 .isin(STATES)]
                     [['year', 'state_name', 'yield_bu_acre']]
                     .reset_index(drop=True))
        state_df.to_csv(STATE_CSV, index=False)
        print(f'  Saved {len(state_df)} rows -> {STATE_CSV}')

        print('Fetching NASS county-level data ...')
        raw_county = _fetch_nass('COUNTY')
        county_df  = (raw_county[raw_county['state_name'].str.upper()
                                   .isin(STATES)]
                      [['year', 'state_name', 'county_name', 'yield_bu_acre']]
                      .reset_index(drop=True))
        county_df.to_csv(COUNTY_CSV, index=False)
        print(f'  Saved {len(county_df)} rows -> {COUNTY_CSV}')
    except Exception as exc:
        raise RuntimeError('Unable to fetch USDA NASS QuickStats data. Fix NASS_API_KEY in your .env or add cached corn_yields_state.csv and corn_yields_county.csv under /Users/jacindabietz/corn_nb/data/nass/.') from exc

print(f'\nState-level rows  : {len(state_df)}')
print(f'County-level rows : {len(county_df)}')

# -- groupby summary --
print('\n-- State-level summary (bu/acre) --')
summary = (state_df.groupby('state_name')['yield_bu_acre']
           .agg(['mean', 'std', 'min', 'max'])
           .round(2))
print(summary.to_string())

# -- Plot A: line chart --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for state, grp in state_df.groupby('state_name'):
    grp_s = grp.sort_values('year')
    ax.plot(grp_s['year'], grp_s['yield_bu_acre'], marker='o', label=state)
ax.set_title('Corn Yield by State 2005-2024')
ax.set_xlabel('Year')
ax.set_ylabel('Yield (bu/acre)')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# -- Plot B: box plot --
ax2 = axes[1]
order  = state_df.groupby('state_name')['yield_bu_acre'].median().sort_values(ascending=False).index.tolist()
data_b = [state_df[state_df['state_name'] == s]['yield_bu_acre'].values for s in order]
ax2.boxplot(data_b, labels=[s[:4] for s in order], patch_artist=True)
ax2.set_title('Yield Distribution by State 2005-2024')
ax2.set_xlabel('State')
ax2.set_ylabel('Yield (bu/acre)')
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4.5 - CDL corn mask: download + cache per state/year (2005-2024)
# Fetch strategy (in order):
#   1. USDA ArcGIS ImageServer (pdi.scinet.usda.gov/image/rest/services/CDL_WM)
#   2. NDVI-threshold proxy mask from the HLS patch (Aug-Oct NDVI > 0.4)
import os, json, requests
import numpy as np, matplotlib.pyplot as plt

CDL_DIR    = '/Users/jacindabietz/corn_nb/data/cdl'
CDL_STATES = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
CDL_YEARS  = list(range(2005, 2025))

STATE_BBOXES_CDL = {
    'Iowa'     : (-96.6, 40.4, -90.1, 43.5),
    'Colorado' : (-109.1, 36.9, -102.0, 41.0),
    'Wisconsin': (-92.9, 42.5, -86.8, 47.1),
    'Missouri' : (-95.8, 35.9, -89.1, 40.6),
    'Nebraska' : (-104.1, 40.0, -95.3, 43.0),
}

TARGET_SHAPE = (64, 64)

CDL_IMAGESERVER = ("https://pdi.scinet.usda.gov/image/rest/services/"
                   "CDL_WM/ImageServer/exportImage")


def _state_slug(state):
    return state.lower().replace(' ', '_')


def _wgs84_to_webmercator(lon, lat):
    import math
    x = lon * 20037508.34 / 180
    y = math.log(math.tan((90 + lat) * math.pi / 360)) / (math.pi / 180)
    y = y * 20037508.34 / 180
    return x, y


def _try_arcgis_imageserver(state, year):
    """
    Fetch CDL corn mask from the USDA ArcGIS ImageServer (CDL_WM).
    Uses exportImage with mosaicRule to pin the year, returns TIFF bytes,
    then thresholds pixel value 1 (corn).
    """
    import tempfile, rasterio
    from rasterio.enums import Resampling

    west, south, east, north = STATE_BBOXES_CDL[state]

    xmin, ymin = _wgs84_to_webmercator(west, south)
    xmax, ymax = _wgs84_to_webmercator(east, north)

    mosaic_rule = json.dumps({
        "mosaicMethod"  : "esriMosaicAttribute",
        "sortField"     : "Year",
        "sortValue"     : str(year),
        "ascending"     : True,
        "mosaicOperation": "MT_FIRST",
        "where"         : f"Year = {year}"
    })

    params = {
        "bbox"      : f"{xmin},{ymin},{xmax},{ymax}",
        "bboxSR"    : 3857,
        "size"      : f"{TARGET_SHAPE[1]},{TARGET_SHAPE[0]}",
        "imageSR"   : 3857,
        "format"    : "tiff",
        "pixelType" : "U8",
        "mosaicRule": mosaic_rule,
        "f"         : "image",
    }

    r = requests.get(CDL_IMAGESERVER, params=params, timeout=120)
    r.raise_for_status()

    ct = r.headers.get("Content-Type", "")
    if "html" in ct or len(r.content) < 500:
        raise RuntimeError(
            f"ImageServer returned unexpected response (ct={ct}, len={len(r.content)})")

    with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as tmp:
        tmp.write(r.content)
        tmp_path = tmp.name
    try:
        with rasterio.open(tmp_path) as ds:
            raw = ds.read(1, out_shape=TARGET_SHAPE,
                          resampling=Resampling.nearest)
    finally:
        os.unlink(tmp_path)

    return (raw == 1).astype(bool)  # CDL value 1 = corn


def _try_ndvi_proxy(state, year):
    """
    NDVI-threshold proxy: load the HLS patch and threshold NDVI > 0.4.
    Corn during grain fill (Aug-Oct) has NDVI > 0.4, separating it from
    urban, water, and barren surfaces.  Scientifically defensible for
    hackathon use — affects only spectral aggregates, not encoder input.
    """
    print(f'  CDL unavailable, using NDVI proxy mask for {state} {year}')
    slug     = f"{_state_slug(state)}_{year}_aug1"
    hls_path = f'/Users/jacindabietz/corn_nb/data/hls/{slug}.npy'
    if not os.path.exists(hls_path):
        for dk in ['sep1', 'oct1', 'eos']:
            alt = f'/Users/jacindabietz/corn_nb/data/hls/{_state_slug(state)}_{year}_{dk}.npy'
            if os.path.exists(alt):
                hls_path = alt
                break
    if not os.path.exists(hls_path):
        print(f'  WARNING: no HLS patch found for {state} {year}, '
              'NDVI proxy falls back to full-patch mask')
        return np.ones(TARGET_SHAPE, dtype=bool)

    patch = np.load(hls_path)  # C x H x W; band index 6 = NDVI
    ndvi  = patch[6]
    return (ndvi > 0.4).astype(bool)


def _get_cdl_mask(state, year):
    slug = _state_slug(state)
    path = os.path.join(CDL_DIR, f'{slug}_{year}_corn_mask.npy')
    if os.path.exists(path):
        return np.load(path)

    mask = None

    # --- Tier 1: ArcGIS ImageServer ---
    try:
        mask = _try_arcgis_imageserver(state, year)
        print(f'  CDL via ArcGIS ImageServer: {state} {year}')
    except Exception as e1:
        print(f'  ArcGIS ImageServer failed ({e1})')

        # --- Tier 2: NDVI proxy ---
        mask = _try_ndvi_proxy(state, year)

    np.save(path, mask)
    return mask


os.makedirs(CDL_DIR, exist_ok=True)

print(f'Building CDL corn masks for {len(CDL_STATES)} states x {len(CDL_YEARS)} years ...')
fetched = skipped = 0
for state in CDL_STATES:
    for year in CDL_YEARS:
        slug = _state_slug(state)
        path = os.path.join(CDL_DIR, f'{slug}_{year}_corn_mask.npy')
        if os.path.exists(path):
            skipped += 1
        else:
            try:
                _get_cdl_mask(state, year)
                fetched += 1
            except Exception as e:
                print(f'  WARN: {state} {year} failed ({e})')

print(f'Done. fetched={fetched}  skipped (cached)={skipped}')

# -- Iowa 2022 preview --
iowa_mask = _get_cdl_mask('Iowa', 2022)
print(f'\nIowa 2022 corn mask shape: {iowa_mask.shape}  '
      f'corn fraction: {iowa_mask.mean():.2%}')
plt.figure(figsize=(4, 4))
plt.imshow(iowa_mask, cmap='Greens', interpolation='nearest')
plt.title('Iowa 2022 CDL Corn Mask (64x64)')
plt.colorbar(label='Corn (1=yes)')
plt.tight_layout()
plt.show()

In [2]:
# Cell 5 -- HLS fetch: Iowa 2022 Aug-1 smoke test
import os, numpy as np, matplotlib.pyplot as plt

BASE_DIR = '/notebooks' if os.path.isdir('/notebooks') and os.access('/notebooks', os.W_OK) \
           else os.path.expanduser('~/corn_nb')

OUT = f'{BASE_DIR}/data/hls/iowa_2022_aug1.npy'
os.makedirs(f'{BASE_DIR}/data/hls', exist_ok=True)

EARTHDATA_USERNAME = os.getenv('EARTHDATA_USERNAME')
EARTHDATA_PASSWORD = os.getenv('EARTHDATA_PASSWORD')

IOWA_BBOX  = (-93.8, 41.9, -93.4, 42.1)  # Story County — ~40x20 km, 1-2 tiles
DATE_FROM  = '2022-07-17'
DATE_TO    = '2022-08-15'

# HLS bands used by Prithvi: B02, B03, B04, B8A, B11, B12
HLS_BANDS  = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
BAND_NAMES = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12', 'NDVI', 'EVI', 'LSWI']
OUT_SHAPE  = (64, 64)


def _fetch_hls():
    import earthaccess, rasterio, tempfile
    from rasterio.enums import Resampling
    from rasterio.merge import merge
    from rasterio.warp import calculate_default_transform, reproject
    from rasterio.io import MemoryFile

    TARGET_CRS = 'EPSG:32615'  # UTM Zone 15N — covers most of Iowa

    earthaccess.login(strategy='environment')
    results = earthaccess.search_data(
        short_name   = 'HLSS30',
        bounding_box = IOWA_BBOX,
        temporal     = (DATE_FROM, DATE_TO),
        count        = 3,
    )
    if not results:
        raise RuntimeError('No HLS HLSS30 granules found for Iowa 2022 Aug window')

    tmp_dir = f'{BASE_DIR}/data/hls/tmp_iowa_2022_aug1'
    os.makedirs(tmp_dir, exist_ok=True)
    files = earthaccess.download(results, tmp_dir)
    print(f'  Downloaded {len(files)} HLS file(s)')

    def reproject_to_target(src_path, target_crs):
        with rasterio.open(str(src_path)) as src:
            transform, width, height = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds
            )
            profile = src.profile.copy()
            profile.update(
                crs=target_crs,
                transform=transform,
                width=width,
                height=height,
                nodata=0,
            )
            mem = MemoryFile()
            with mem.open(**profile) as dst:
                for i in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=target_crs,
                        resampling=Resampling.bilinear,
                    )
            return mem  # caller must keep this alive until merge is done

    band_arrays = []
    for band in HLS_BANDS:
        band_files = [
            f for f in files
            if str(f).endswith(f'.{band}.tif') or str(f).endswith(f'_{band}.tif')
        ]

        if band_files:
            # Reproject every granule to the common CRS before merging
            mem_files = [reproject_to_target(f, TARGET_CRS) for f in band_files]
            datasets  = [mf.open() for mf in mem_files]
            try:
                mosaic, _ = merge(datasets, nodata=0)
                raw_band  = mosaic[0].astype(np.float32)
            finally:
                for ds in datasets:
                    ds.close()
                for mf in mem_files:
                    mf.close()
        else:
            with rasterio.open(str(files[0])) as ds:
                band_idx = HLS_BANDS.index(band) + 1
                raw_band = ds.read(band_idx).astype(np.float32)

        raw_band /= 10000.0
        from skimage.transform import resize as sk_resize
        arr = sk_resize(raw_band, OUT_SHAPE, order=1,
                        mode='reflect', anti_aliasing=True).astype(np.float32)
        band_arrays.append(arr)

    raw = np.stack(band_arrays, axis=0)
    b02, b03, b04, b8a, b11, b12 = raw
    nodata = (b02 == 0) & (b04 == 0) & (b8a == 0)
    ndvi = np.where(nodata, 0.0, (b8a - b04) / (b8a + b04 + 1e-9))
    evi  = np.where(nodata, 0.0,
                    2.5 * (b8a - b04) / (b8a + 6*b04 - 7.5*b02 + 1 + 1e-9))
    lswi = np.where(nodata, 0.0, (b8a - b11) / (b8a + b11 + 1e-9))

    return np.concatenate(
        [raw, ndvi[None], evi[None], lswi[None]], axis=0
    ).astype(np.float32)


def _fetch_planetary_computer():
    import pystac_client, planetary_computer, rasterio
    from rasterio.enums import Resampling
    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace,
    )
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=IOWA_BBOX,
        datetime=f'{DATE_FROM}/{DATE_TO}',
        query={'eo:cloud_cover': {'lt': 30}},
        max_items=5,
    )
    items = list(search.items())
    if not items:
        raise RuntimeError('No Sentinel-2 items found on Planetary Computer')
    item = items[0]
    print(f'  MPC fallback item: {item.id}')

    # Map HLS band names to S2 L2A asset keys
    band_map = {'B02': 'B02', 'B03': 'B03', 'B04': 'B04',
                'B8A': 'B8A', 'B11': 'B11', 'B12': 'B12'}
    arrays = []
    for band in HLS_BANDS:
        href = item.assets[band_map[band]].href
        with rasterio.open(href) as ds:
            arr = ds.read(
                1,
                out_shape=OUT_SHAPE,
                resampling=Resampling.bilinear,
            ).astype(np.float32) / 10000.0
        arrays.append(arr)

    raw = np.stack(arrays, axis=0)
    b02, b03, b04, b8a, b11, b12 = raw
    nodata = (b02 == 0) & (b04 == 0) & (b8a == 0)
    ndvi = np.where(nodata, 0.0, (b8a - b04) / (b8a + b04 + 1e-9))
    evi  = np.where(nodata, 0.0, 2.5 * (b8a - b04) / (b8a + 6*b04 - 7.5*b02 + 1 + 1e-9))
    lswi = np.where(nodata, 0.0, (b8a - b11) / (b8a + b11 + 1e-9))
    return np.concatenate([raw, ndvi[None], evi[None], lswi[None]],
                          axis=0).astype(np.float32)


if os.path.exists(OUT):
    patch = np.load(OUT)
    print(f'Loaded cached patch: {patch.shape}')
else:
    print('Fetching from NASA HLS (HLSS30) ...')
    try:
        patch = _fetch_hls()
    except Exception as e:
        print(f'  HLS failed ({e}), falling back to Planetary Computer ...')
        patch = _fetch_planetary_computer()
    np.save(OUT, patch)
    print(f'Saved -> {OUT}')

print(f'Shape: {patch.shape}  (C x H x W)')
for i, name in enumerate(BAND_NAMES):
    print(f'  {name:6s}: min={patch[i].min():.4f}  max={patch[i].max():.4f}')


def stretch(arr):
    lo, hi = np.percentile(arr, 2), np.percentile(arr, 98)
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

rgb = np.stack([stretch(patch[2]), stretch(patch[1]), stretch(patch[0])], axis=-1)
plt.figure(figsize=(6, 4))
plt.imshow(rgb)
plt.title('Iowa 2022 Aug-1 -- HLS RGB (B04/B03/B02)')
plt.axis('off')
plt.tight_layout()
plt.show()


Fetching from NASA HLS (HLSS30) ...
  HLS failed (No module named 'earthaccess'), falling back to Planetary Computer ...


ModuleNotFoundError: No module named 'pystac_client'

In [ ]:
# Cell 6 -- NAIP fetch: Story County Iowa (with fallback year)
import os, numpy as np, matplotlib.pyplot as plt

OUT = '/Users/jacindabietz/corn_nb/data/naip/iowa_story_2020.npy'
os.makedirs('/Users/jacindabietz/corn_nb/data/naip', exist_ok=True)

STORY_BBOX   = (-93.8, 41.9, -93.4, 42.1)
TARGET_SHAPE = (4, 224, 224)

if os.path.exists(OUT):
    patch = np.load(OUT)
    print(f'Loaded cached NAIP patch: {patch.shape}')
else:
    import pystac_client, planetary_computer, rasterio
    from rasterio.enums import Resampling

    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace,
    )

    # --- Step 1: confirm 'naip' is a valid collection ---
    try:
        col = catalog.get_collection('naip')
        print(f'Collection found: {col.id}  |  extent: {col.extent.temporal.intervals}')
    except Exception as e:
        print(f'Collection lookup failed: {e}')

    # --- Step 2: try multiple years in case 2020 isn't available ---
    item = None
    for year in [2020, 2021, 2019, 2018]:
        dt = f'{year}-01-01/{year}-12-31'
        search = catalog.search(
            collections=['naip'],
            bbox=STORY_BBOX,
            datetime=dt,
            max_items=10,
        )
        items = list(search.items())
        print(f'{year}: {len(items)} items found')
        if items:
            item = planetary_computer.sign(items[0])
            print(f'Using item: {item.id}  (year={year})')
            break

    if item is None:
        # --- Step 3: widen bbox to all of Iowa as last resort ---
        print('No items in Story County bbox — widening to full Iowa bbox ...')
        IOWA_BBOX = (-96.7, 40.3, -90.1, 43.5)
        search = catalog.search(
            collections=['naip'],
            bbox=IOWA_BBOX,
            datetime='2018-01-01/2021-12-31',
            max_items=5,
        )
        items = list(search.items())
        print(f'Iowa-wide search: {len(items)} items')
        if not items:
            raise RuntimeError(
                'NAIP unavailable on Planetary Computer for Iowa. '
                'Check https://planetarycomputer.microsoft.com/dataset/naip '
                'for available states/years.'
            )
        item = planetary_computer.sign(items[0])
        print(f'Fallback item: {item.id}')

    href = item.assets['image'].href
    with rasterio.open(href) as ds:
        # NAIP is RGBI (4-band); read all 4
        n_bands = min(ds.count, 4)
        out_shape = (n_bands, TARGET_SHAPE[1], TARGET_SHAPE[2])
        data = ds.read(
            out_shape=out_shape,
            resampling=Resampling.bilinear,
        ).astype(np.float32)

    if data.max() > 1.0:
        data = data / 255.0

    # Pad to 4 bands if tile only has 3
    if data.shape[0] < 4:
        pad = np.zeros((4 - data.shape[0], *data.shape[1:]), dtype=np.float32)
        data = np.concatenate([data, pad], axis=0)

    patch = data
    np.save(OUT, patch)
    print(f'Saved -> {OUT}')

print(f'Shape: {patch.shape}  (4 x 224 x 224 -- R,G,B,NIR)')
for i, name in enumerate(['R', 'G', 'B', 'NIR']):
    print(f'  {name}: min={patch[i].min():.4f}  max={patch[i].max():.4f}')

def stretch(arr):
    lo, hi = np.percentile(arr, 2), np.percentile(arr, 98)
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

rgb = np.stack([stretch(patch[0]), stretch(patch[1]), stretch(patch[2])], axis=-1)
plt.figure(figsize=(5, 5))
plt.imshow(rgb)
plt.title('NAIP -- Story County, Iowa (RGB)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6b -- Generate synthetic Sentinel-2 patch for testing
import numpy as np, os

os.makedirs('/Users/jacindabietz/corn_nb/data/sentinel', exist_ok=True)
SENTINEL_PATH = '/Users/jacindabietz/corn_nb/data/sentinel/iowa_2022_aug1.npy'

if not os.path.exists(SENTINEL_PATH):
    # 9 bands, 224x224 patch -- realistic value ranges for Sentinel-2 surface reflectance
    rng = np.random.default_rng(42)
    patch = rng.uniform(0.0, 0.3, size=(9, 224, 224)).astype(np.float32)
    np.save(SENTINEL_PATH, patch)
    print(f'Synthetic patch saved: {patch.shape}')
else:
    print('Sentinel patch already exists, skipping.')

In [ ]:
# Cell 7-debug-2
from huggingface_hub import hf_hub_download

MODEL_ID = 'ibm-nasa-geospatial/Prithvi-100M'

print("=== prithvi_mae.py top 80 lines ===")
mae_py = hf_hub_download(repo_id=MODEL_ID, filename='prithvi_mae.py')
with open(mae_py) as f:
    lines = f.readlines()
print(''.join(lines[:80]))
print(f"\n(prithvi_mae.py is {len(lines)} lines total)")

print("\n=== Class definitions ===")
for i, line in enumerate(lines):
    if line.startswith('class '):
        print(f"  line {i}: {line.strip()}")

print("\n=== Method definitions (def ...) ===")
for i, line in enumerate(lines):
    if line.startswith('    def '):
        print(f"  line {i}: {line.strip()}")

print("\n=== inference.py top 80 lines ===")
inf_py = hf_hub_download(repo_id=MODEL_ID, filename='inference.py')
with open(inf_py) as f:
    lines2 = f.readlines()
print(''.join(lines2[:80]))

In [ ]:
%pip install einops -q

import os
emb = "/Users/jacindabietz/corn_nb/data/embeddings/iowa_2022_aug1.npy"
if os.path.exists(emb):
    os.remove(emb)


In [ ]:
from huggingface_hub import hf_hub_download
import importlib.util, sys

mae_py = hf_hub_download(repo_id='ibm-nasa-geospatial/Prithvi-100M', filename='prithvi_mae.py')
spec   = importlib.util.spec_from_file_location('prithvi_mae', mae_py)
module = importlib.util.module_from_spec(spec)
sys.modules['prithvi_mae'] = module
spec.loader.exec_module(module)

import inspect
print(inspect.getsource(module.PrithviViT.forward_features))

In [ ]:
# Cell 7 -- Prithvi-100M encoder: load + smoke test on Iowa 2022 Aug-1 patch
import os, numpy as np, torch, importlib.util, sys, json
import torch.nn.functional as F

SENTINEL_PATH  = '/Users/jacindabietz/corn_nb/data/sentinel/iowa_2022_aug1.npy'
EMBEDDING_PATH = '/Users/jacindabietz/corn_nb/data/embeddings/iowa_2022_aug1.npy'
os.makedirs('/Users/jacindabietz/corn_nb/data/embeddings', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'ibm-nasa-geospatial/Prithvi-100M'

PRITHVI_MEAN = np.array([775.229, 1080.993, 1228.586, 2497.202, 2204.214, 1610.832],
                         dtype=np.float32)
PRITHVI_STD  = np.array([1281.526, 1270.030, 1399.480, 1368.345, 1291.676, 1154.506],
                         dtype=np.float32)

def load_prithvi():
    from huggingface_hub import hf_hub_download
    mae_py    = hf_hub_download(repo_id=MODEL_ID, filename='prithvi_mae.py')
    ckpt_path = hf_hub_download(repo_id=MODEL_ID, filename='Prithvi_100M.pt')
    cfg_path  = hf_hub_download(repo_id=MODEL_ID, filename='config.json')

    with open(cfg_path) as f:
        cfg = json.load(f)
    pcfg = cfg['pretrained_cfg']

    spec   = importlib.util.spec_from_file_location('prithvi_mae', mae_py)
    module = importlib.util.module_from_spec(spec)
    sys.modules['prithvi_mae'] = module
    spec.loader.exec_module(module)

    model = module.PrithviMAE(
        img_size          = pcfg['img_size'],
        patch_size        = pcfg['patch_size'],
        num_frames        = pcfg['num_frames'],
        in_chans          = pcfg['in_chans'],
        embed_dim         = pcfg['embed_dim'],
        depth             = pcfg['depth'],
        num_heads         = pcfg['num_heads'],
        decoder_embed_dim = pcfg['decoder_embed_dim'],
        decoder_depth     = pcfg['decoder_depth'],
        decoder_num_heads = pcfg['decoder_num_heads'],
        mlp_ratio         = pcfg['mlp_ratio'],
    )

    state_dict = torch.load(ckpt_path, map_location='cpu')
    if isinstance(state_dict, dict) and 'model' in state_dict:
        state_dict = state_dict['model']
    model.load_state_dict(state_dict, strict=False)
    model.eval().to(DEVICE)
    return model

def encode_prithvi(model, patch):
    p = patch[:6].copy()
    p = (p - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
    t = torch.from_numpy(p).float().unsqueeze(0)          # 1 x 6 x H x W
    t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
    t = t.unsqueeze(2).repeat(1, 1, 3, 1, 1).to(DEVICE)  # 1 x 6 x 3 x 224 x 224
    with torch.no_grad():
        feats = model.encoder.forward_features(t)
    last = feats[-1]                   # 1 x (1+n_patches) x 768
    emb  = last[:, 1:, :].mean(dim=1) # 1 x 768
    return emb.squeeze().cpu().numpy().flatten().astype(np.float32)

# -- delete stale ResNet cache if present --
if os.path.exists(EMBEDDING_PATH):
    cached = np.load(EMBEDDING_PATH)
    if cached.shape != (768,):
        print(f'Stale cache found (shape={cached.shape}), deleting ...')
        os.remove(EMBEDDING_PATH)
    else:
        print(f'Valid Prithvi cache found: {cached.shape}')

if os.path.exists(EMBEDDING_PATH):
    embedding = np.load(EMBEDDING_PATH)
    print(f'Loaded cached embedding: {embedding.shape}')
else:
    raw = np.load(SENTINEL_PATH)   # C x H x W
    print(f'Loaded patch: {raw.shape}')
    print(f'Loading {MODEL_ID} ...')
    try:
        model    = load_prithvi()
        embedding = encode_prithvi(model, raw)
        print(f'Prithvi embedding shape: {embedding.shape}')
    except Exception as e:
        print(f'Prithvi failed ({e})\nFalling back to ResNet-50 ...')
        import torch.nn as nn
        from torchvision import models
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        backbone.fc = nn.Identity()
        in_channels = raw.shape[0]
        adapter = nn.Conv2d(in_channels, 3, kernel_size=1, bias=False)
        nn.init.kaiming_normal_(adapter.weight)
        enc = nn.Sequential(adapter, backbone).eval().to(DEVICE)
        patch = torch.from_numpy(raw).float().unsqueeze(0)
        patch = F.interpolate(patch, size=(224, 224), mode='bilinear',
                              align_corners=False).to(DEVICE)
        with torch.no_grad():
            feat = enc(patch)
        embedding = feat.squeeze().cpu().numpy().flatten().astype(np.float32)
        print(f'ResNet-50 fallback embedding shape: {embedding.shape}')

    np.save(EMBEDDING_PATH, embedding)
    print(f'Saved -> {EMBEDDING_PATH}')

print(f'\nEmbedding shape : {embedding.shape}')
print(f'min={embedding.min():.4f}  max={embedding.max():.4f}  mean={embedding.mean():.4f}')

In [ ]:
import os, numpy as np, glob

stale = [p for p in glob.glob('/Users/jacindabietz/corn_nb/data/embeddings/*.npy')
         if np.load(p).shape != (768,)]
print(f'Deleting {len(stale)} stale embeddings ...')
for p in stale:
    os.remove(p)
print('Done.')

import os, glob

# Remove patches from 2015 onward that were fetched via Planetary Computer
# instead of HLS due to the auth bug
for state in ['iowa', 'colorado', 'wisconsin', 'missouri', 'nebraska']:
    for year in range(2015, 2025):
        for date_key in ['aug1', 'sep1', 'oct1', 'eos']:
            slug   = f'{state}_{year}_{date_key}'
            s_path = f'/Users/jacindabietz/corn_nb/data/hls/{slug}.npy'
            e_path = f'/Users/jacindabietz/corn_nb/data/embeddings/{slug}.npy'
            for p in [s_path, e_path]:
                if os.path.exists(p):
                    os.remove(p)
                    print(f'Removed: {p}')

In [ ]:
# Cell 8 -- Build full feature matrix X, labels y, metadata
import os, json, time, importlib.util, sys, earthaccess
import numpy as np, pandas as pd, requests, torch
import torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor, as_completed

X_PATH    = '/Users/jacindabietz/corn_nb/data/features/X.npy'
Y_PATH    = '/Users/jacindabietz/corn_nb/data/features/y.npy'
META_PATH = '/Users/jacindabietz/corn_nb/data/features/metadata.csv'
os.makedirs('/Users/jacindabietz/corn_nb/data/features',   exist_ok=True)
os.makedirs('/Users/jacindabietz/corn_nb/data/hls',        exist_ok=True)
os.makedirs('/Users/jacindabietz/corn_nb/data/embeddings', exist_ok=True)
os.makedirs('/Users/jacindabietz/corn_nb/data/weather',    exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

PRITHVI_MEAN = np.array([775.229, 1080.993, 1228.586, 2497.202, 2204.214, 1610.832],
                         dtype=np.float32)
PRITHVI_STD  = np.array([1281.526, 1270.030, 1399.480, 1368.345, 1291.676, 1154.506],
                         dtype=np.float32)

STATES = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
YEARS  = list(range(2005, 2025))
DATES  = ['aug1', 'sep1', 'oct1', 'eos']

DATE_WINDOWS = {
    'aug1': ('07-17', '08-15'),
    'sep1': ('08-17', '09-15'),
    'oct1': ('09-17', '10-15'),
    'eos' : ('10-17', '10-31'),
}

STATE_BBOXES = {
    'Iowa'     : (-96.6, 40.4, -90.1, 43.5),
    'Colorado' : (-109.1, 36.9, -102.0, 41.0),
    'Wisconsin': (-92.9, 42.5, -86.8, 47.1),
    'Missouri' : (-95.8, 35.9, -89.1, 40.6),
    'Nebraska' : (-104.1, 40.0, -95.3, 43.0),
}

STATE_CENTROIDS = {
    'Iowa'     : (42.0, -93.5),
    'Colorado' : (39.0, -105.5),
    'Wisconsin': (44.5, -89.5),
    'Missouri' : (38.3, -92.5),
    'Nebraska' : (41.5, -99.9),
}

HLS_BANDS = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
OUT_SHAPE  = (64, 64)

SH_CLIENT_ID     = os.getenv('SH_CLIENT_ID')
SH_CLIENT_SECRET = os.getenv('SH_CLIENT_SECRET')

EARTHDATA_USER  = os.getenv('EARTHDATA_USERNAME')
EARTHDATA_PASS  = os.getenv('EARTHDATA_PASSWORD')
EARTHDATA_TOKEN = os.getenv('EARTHDATA_TOKEN')

try:
    if EARTHDATA_TOKEN:
        earthaccess.login(strategy='token')
    elif EARTHDATA_USER and EARTHDATA_PASS:
        earthaccess.login(strategy='environment')
    else:
        print('No Earthdata credentials found in env; HLS will fall back to Planetary Computer when needed.')
    print(f'Earthdata login OK: {earthaccess.auth.Auth().authenticated}')
except Exception as e:
    print(f'Earthdata login failed ({e}) -- HLS will fall back to Planetary Computer')

if os.path.exists(X_PATH) and os.path.exists(Y_PATH) and os.path.exists(META_PATH):
    X    = np.load(X_PATH)
    y    = np.load(Y_PATH)
    meta = pd.read_csv(META_PATH)
    print(f'Loaded cached feature matrix: X={X.shape}, y={y.shape}')
    print(meta.head())
else:
    nass_path = '/Users/jacindabietz/corn_nb/data/nass/corn_yields_state.csv'
    if not os.path.exists(nass_path):
        raise FileNotFoundError('corn_yields_state.csv not found -- run Cell 4 first.')
    nass_df = pd.read_csv(nass_path)
    nass_lookup = {
        (str(row['state_name']).title(), int(row['year'])): float(row['yield_bu_acre'])
        for _, row in nass_df.iterrows()
    }

    # ── encoder ───────────────────────────────────────────────────────────────
    _encoder      = None
    _encoder_type = None

    def get_encoder():
        global _encoder, _encoder_type
        if _encoder is not None:
            return _encoder, _encoder_type

        try:
            import importlib.util, sys, json
            from huggingface_hub import hf_hub_download

            print('Loading Prithvi-100M encoder ...')
            MODEL_ID  = 'ibm-nasa-geospatial/Prithvi-100M'
            mae_py    = hf_hub_download(repo_id=MODEL_ID, filename='prithvi_mae.py')
            ckpt_path = hf_hub_download(repo_id=MODEL_ID, filename='Prithvi_100M.pt')
            cfg_path  = hf_hub_download(repo_id=MODEL_ID, filename='config.json')

            with open(cfg_path) as f:
                cfg = json.load(f)
            pcfg = cfg['pretrained_cfg']

            spec   = importlib.util.spec_from_file_location('prithvi_mae', mae_py)
            module = importlib.util.module_from_spec(spec)
            sys.modules['prithvi_mae'] = module
            spec.loader.exec_module(module)

            model = module.PrithviMAE(
                img_size          = pcfg['img_size'],
                patch_size        = pcfg['patch_size'],
                num_frames        = pcfg['num_frames'],
                in_chans          = pcfg['in_chans'],
                embed_dim         = pcfg['embed_dim'],
                depth             = pcfg['depth'],
                num_heads         = pcfg['num_heads'],
                decoder_embed_dim = pcfg['decoder_embed_dim'],
                decoder_depth     = pcfg['decoder_depth'],
                decoder_num_heads = pcfg['decoder_num_heads'],
                mlp_ratio         = pcfg['mlp_ratio'],
            )
            state_dict = torch.load(ckpt_path, map_location='cpu')
            if isinstance(state_dict, dict) and 'model' in state_dict:
                state_dict = state_dict['model']
            model.load_state_dict(state_dict, strict=False)
            model.eval().to(DEVICE)
            _encoder, _encoder_type = model, 'prithvi'
            print(f'Prithvi-100M loaded on {DEVICE}.')

        except Exception as e:
            print(f'Prithvi failed ({e}), using ResNet-50 fallback.')
            import torch.nn as nn
            from torchvision import models
            backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            backbone.fc = nn.Identity()
            adapter = nn.Conv2d(9, 3, kernel_size=1, bias=False)
            nn.init.kaiming_normal_(adapter.weight)
            m = nn.Sequential(adapter, backbone).eval().to(DEVICE)
            _encoder, _encoder_type = m, 'resnet'

        return _encoder, _encoder_type

    # ── HLS via earthaccess ───────────────────────────────────────────────────
    def _fetch_hls(state, year, date_key):
        import rasterio, glob
        from rasterio.enums import Resampling

        from_str, to_str = DATE_WINDOWS[date_key]
        bbox_coords      = STATE_BBOXES[state]
        results = earthaccess.search_data(
            short_name   = 'HLSS30',
            bounding_box = bbox_coords,
            temporal     = (f'{year}-{from_str}', f'{year}-{to_str}'),
            count        = 5,
        )
        if not results:
            raise RuntimeError(f'No HLS granules for {state} {year} {date_key}')

        slug    = f"{state.lower().replace(' ', '_')}_{year}_{date_key}"
        tmp_dir = f'/Users/jacindabietz/corn_nb/data/hls/tmp_{slug}'
        os.makedirs(tmp_dir, exist_ok=True)

        raw_files = earthaccess.download(results[:1], tmp_dir)

        print(f'    raw_files types : {[type(f).__name__ for f in raw_files]}')
        print(f'    raw_files values: {raw_files}')

        files = [str(f) for f in raw_files]

        all_in_dir = glob.glob(f'{tmp_dir}/**/*', recursive=True)
        all_in_dir = [f for f in all_in_dir if os.path.isfile(f)]
        print(f'    all files in tmp_dir: {all_in_dir}')

        files = list(set(files + all_in_dir))
        print(f'    merged file list: {files}')

        if not files:
            raise RuntimeError(f'Download returned no files for {slug}')

        tif_files = [f for f in files if f.lower().endswith('.tif')]
        print(f'    .tif files found: {tif_files}')

        band_arrays = []
        for band in HLS_BANDS:
            band_file = next(
                (f for f in tif_files if f.upper().endswith(f'.{band.upper()}.TIF')),
                None,
            )
            print(f'    band {band} -> matched file: {band_file}')

            if band_file:
                with rasterio.open(band_file) as ds:
                    arr = ds.read(
                        1, out_shape=OUT_SHAPE,
                        resampling=Resampling.bilinear,
                    ).astype(np.float32) / 10000.0
            elif tif_files:
                band_idx = HLS_BANDS.index(band) + 1
                with rasterio.open(tif_files[0]) as ds:
                    print(f'      multi-band fallback: {ds.count} bands available, reading idx {min(band_idx, ds.count)}')
                    idx = min(band_idx, ds.count)
                    arr = ds.read(
                        idx, out_shape=OUT_SHAPE,
                        resampling=Resampling.bilinear,
                    ).astype(np.float32) / 10000.0
            else:
                print(f'      WARNING: no tif at all for {band}, using zeros')
                arr = np.zeros(OUT_SHAPE, dtype=np.float32)

            band_arrays.append(arr)

        raw = np.stack(band_arrays, axis=0)
        b02, b03, b04, b8a, b11, b12 = raw
        ndvi = (b8a - b04) / (b8a + b04 + 1e-9)
        evi  = 2.5 * (b8a - b04) / (b8a + 6*b04 - 7.5*b02 + 1 + 1e-9)
        lswi = (b8a - b11) / (b8a + b11 + 1e-9)
        return np.concatenate([raw, ndvi[None], evi[None], lswi[None]],
                              axis=0).astype(np.float32)

    # ── Planetary Computer ────────────────────────────────────────────────────
    def _fetch_planetary_computer(state, year, date_key):
        import pystac_client, planetary_computer, rasterio
        from rasterio.enums import Resampling

        from_str, to_str = DATE_WINDOWS[date_key]
        bbox_coords      = STATE_BBOXES[state]
        dt_range         = f'{year}-{from_str}/{year}-{to_str}'

        catalog = pystac_client.Client.open(
            'https://planetarycomputer.microsoft.com/api/stac/v1',
            modifier=planetary_computer.sign_inplace,
        )

        items = []
        for collection in ['sentinel-2-l2a', 'landsat-c2-l2']:
            try:
                search = catalog.search(
                    collections = [collection],
                    bbox        = bbox_coords,
                    datetime    = dt_range,
                    filter      = {'op': 'lte',
                                   'args': [{'property': 'eo:cloud_cover'}, 40]},
                    filter_lang = 'cql2-json',
                    max_items   = 10,
                )
                items = list(search.items())
            except Exception:
                search = catalog.search(
                    collections = [collection],
                    bbox        = bbox_coords,
                    datetime    = dt_range,
                    max_items   = 20,
                )
                items = [
                    i for i in search.items()
                    if i.properties.get('eo:cloud_cover', 100) < 40
                ]

            if items:
                break

        if not items:
            raise RuntimeError(
                f'No satellite items found for {state} {year} {date_key} '
                f'(tried sentinel-2-l2a and landsat-c2-l2)'
            )

        item = items[0]
        collection_id = item.collection_id

        if 'sentinel' in collection_id:
            band_map = {'B02': 'B02', 'B03': 'B03', 'B04': 'B04',
                        'B8A': 'B8A', 'B11': 'B11', 'B12': 'B12'}
        else:
            band_map = {'B02': 'blue', 'B03': 'green', 'B04': 'red',
                        'B8A': 'nir08', 'B11': 'swir16', 'B12': 'swir22'}

        arrays = []
        for band in HLS_BANDS:
            asset_key = band_map[band]
            if asset_key not in item.assets:
                arrays.append(np.zeros(OUT_SHAPE, dtype=np.float32))
                continue
            href = item.assets[asset_key].href
            with rasterio.open(href) as ds:
                arr = ds.read(1, out_shape=OUT_SHAPE,
                              resampling=Resampling.bilinear,
                              ).astype(np.float32) / 10000.0
            arrays.append(arr)

        raw = np.stack(arrays, axis=0)
        b02, b03, b04, b8a, b11, b12 = raw
        ndvi = (b8a - b04) / (b8a + b04 + 1e-9)
        evi  = 2.5 * (b8a - b04) / (b8a + 6*b04 - 7.5*b02 + 1 + 1e-9)
        lswi = (b8a - b11) / (b8a + b11 + 1e-9)
        return np.concatenate([raw, ndvi[None], evi[None], lswi[None]], axis=0).astype(np.float32)

    # ── Open-Meteo synthetic fallback ─────────────────────────────────────────
    def _fetch_open_meteo(state, year, date_key):
        from_str, to_str = DATE_WINDOWS[date_key]
        lat, lon         = STATE_CENTROIDS[state]
        url = 'https://archive-api.open-meteo.com/v1/archive'
        params = {
            'latitude'         : lat,
            'longitude'        : lon,
            'start_date'       : f'{year}-{from_str}',
            'end_date'         : f'{year}-{to_str}',
            'daily'            : ('temperature_2m_max,temperature_2m_min,'
                                  'precipitation_sum,shortwave_radiation_sum,'
                                  'et0_fao_evapotranspiration'),
            'hourly'           : 'soil_moisture_0_to_7cm',
            'timezone'         : 'America/Chicago',
        }
        for attempt in range(3):
            try:
                r = requests.get(url, params=params, timeout=60)
                r.raise_for_status()
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(5 * (attempt + 1))

        daily  = r.json()['daily']
        tmax   = np.array(daily['temperature_2m_max'],  dtype=np.float32)
        tmin   = np.array(daily['temperature_2m_min'],  dtype=np.float32)
        precip = np.array(daily['precipitation_sum'],   dtype=np.float32)
        rad    = np.array(daily['shortwave_radiation_sum'], dtype=np.float32)

        H, W = OUT_SHAPE
        norm = lambda x: ((x - x.mean()) / (x.std() + 1e-9))
        bands = [
            np.full((H, W), float(norm(tmax).mean()),   dtype=np.float32),
            np.full((H, W), float(norm(tmin).mean()),   dtype=np.float32),
            np.full((H, W), float(norm(precip).mean()), dtype=np.float32),
            np.full((H, W), float(norm(rad).mean()),    dtype=np.float32),
            np.full((H, W), float(tmax.mean() / 40.0), dtype=np.float32),
            np.full((H, W), float(precip.sum() / 500.), dtype=np.float32),
            np.full((H, W), float(np.clip((tmax + tmin) / 2 - 10, 0, None).mean() / 30.), dtype=np.float32),
            np.full((H, W), 0.0, dtype=np.float32),
            np.full((H, W), 0.0, dtype=np.float32),
        ]
        return np.stack(bands, axis=0).astype(np.float32)

    def fetch_patch(state, year, date_key):
        try:
            return _fetch_hls(state, year, date_key)
        except Exception as e1:
            print(f'    HLS failed ({type(e1).__name__}: {e1})')
            try:
                patch = _fetch_planetary_computer(state, year, date_key)
                print(f'    Planetary Computer OK')
                return patch
            except Exception as e2:
                print(f'    Planetary Computer failed ({type(e2).__name__}: {e2})')
                print(f'    Using Open-Meteo synthetic patch (weather-only signal)')
                return _fetch_open_meteo(state, year, date_key)

    def _load_cdl_mask(state, year):
        slug = state.lower().replace(' ', '_')
        path = f'/Users/jacindabietz/corn_nb/data/cdl/{slug}_{year}_corn_mask.npy'
        if os.path.exists(path):
            return np.load(path)
        return None

    def get_weather_features(state, year):
        wpath = f'/Users/jacindabietz/corn_nb/data/weather/{state}_{year}.csv'
        if os.path.exists(wpath):
            wdf = pd.read_csv(wpath)
            if 'gdd' not in wdf.columns or 'soil_moisture' not in wdf.columns:
                os.remove(wpath)  # stale cache, re-fetch with new features
            else:
                return np.array([
                    float(wdf['tmax'].mean()),       float(wdf['tmin'].mean()),
                    float(wdf['precip'].sum()),      float(wdf['tmax'].max()),
                    float(wdf['tmin'].min()),        float(wdf['precip'].std()),
                    float(wdf['gdd'].sum()),         float(wdf['soil_moisture'].mean()),
                ], dtype=np.float32)

        lat, lon = STATE_CENTROIDS[state]
        try:
            url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
            params = {
                'parameters': 'T2M_MAX,T2M_MIN,PRECTOTCORR,GWETROOT',
                'community' : 'AG',
                'longitude' : lon,
                'latitude'  : lat,
                'start'     : f'{year}0401',
                'end'       : f'{year}1031',
                'format'    : 'JSON',
            }
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            pdata  = r.json()['properties']['parameter']
            dates  = sorted(pdata['T2M_MAX'].keys())
            tmax       = np.array([pdata['T2M_MAX'][d]         for d in dates], dtype=np.float32)
            tmin       = np.array([pdata['T2M_MIN'][d]         for d in dates], dtype=np.float32)
            precip     = np.array([pdata['PRECTOTCORR'][d]     for d in dates], dtype=np.float32)
            soil_moist = np.array([pdata['GWETROOT'].get(d, np.nan) for d in dates], dtype=np.float32)
        except Exception as e:
            print(f'    NASA POWER failed ({e}), falling back to Open-Meteo ...')
            url = 'https://archive-api.open-meteo.com/v1/archive'
            params = {
                'latitude'  : lat,
                'longitude' : lon,
                'start_date': f'{year}-04-01',
                'end_date'  : f'{year}-10-31',
                'daily'     : 'temperature_2m_max,temperature_2m_min,precipitation_sum,soil_moisture_0_to_7cm',
                'timezone'  : 'America/Chicago',
            }
            for attempt in range(3):
                try:
                    r = requests.get(url, params=params, timeout=30)
                    r.raise_for_status()
                    break
                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(5)
            daily  = r.json()['daily']
            tmax       = np.array(daily['temperature_2m_max'],       dtype=np.float32)
            tmin       = np.array(daily['temperature_2m_min'],       dtype=np.float32)
            precip     = np.array(daily['precipitation_sum'],        dtype=np.float32)
            soil_moist = np.array(daily['soil_moisture_0_to_7cm'],   dtype=np.float32)

        tmax[tmax < -900]          = np.nan
        tmin[tmin < -900]          = np.nan
        precip[precip < 0]         = 0.0
        soil_moist[soil_moist < 0] = np.nan

        # GDD base-10 (standard for corn): daily accumulation of max(0, tmean - 10)
        gdd = np.maximum(0.0, (tmax + tmin) / 2.0 - 10.0)

        wdf = pd.DataFrame({'tmax': tmax, 'tmin': tmin, 'precip': precip,
                             'gdd': gdd, 'soil_moisture': soil_moist})
        wdf.to_csv(wpath, index=False)
        return np.array([
            float(np.nanmean(tmax)),         float(np.nanmean(tmin)),
            float(np.nansum(precip)),        float(np.nanmax(tmax)),
            float(np.nanmin(tmin)),          float(np.nanstd(precip)),
            float(np.nansum(gdd)),           float(np.nanmean(soil_moist)),
        ], dtype=np.float32)

    # ── PHASE 1: parallel weather fetch ───────────────────────────────────────
    print('Pre-fetching weather data (parallel) ...')
    weather_cache = {}
    weather_keys  = [(s, y) for s in STATES for y in YEARS]
    with ThreadPoolExecutor(max_workers=8) as pool:
        futs = {pool.submit(get_weather_features, s, y): (s, y) for s, y in weather_keys}
        for fut in as_completed(futs):
            s, y = futs[fut]
            try:
                weather_cache[(s, y)] = fut.result()
                print(f'  weather OK: {s} {y}')
            except Exception as e:
                print(f'  weather FAILED: {s} {y}: {e}')

    # ── PHASE 2: parallel satellite patch fetch ───────────────────────────────
    def _fetch_and_save_patch(state, year, date_key):
        slug   = f"{state.lower().replace(' ','_')}_{year}_{date_key}"
        s_path = f'/Users/jacindabietz/corn_nb/data/hls/{slug}.npy'
        if os.path.exists(s_path):
            return slug, 'cached'
        patch = fetch_patch(state, year, date_key)
        np.save(s_path, patch)
        return slug, 'fetched'

    all_tasks = [(s, y, d) for s in STATES for y in YEARS for d in DATES]
    print(f'\nFetching {len(all_tasks)} satellite patches (parallel) ...')
    fetch_done = 0
    with ThreadPoolExecutor(max_workers=6) as pool:
        futs = {pool.submit(_fetch_and_save_patch, s, y, d): (s, y, d)
                for s, y, d in all_tasks}
        for fut in as_completed(futs):
            fetch_done += 1
            try:
                slug, status = fut.result()
                if status == 'fetched':
                    print(f'  fetched: {slug}')
            except Exception as e:
                s, y, d = futs[fut]
                print(f'  FAILED: {s} {y} {d}: {e}')
            if fetch_done % 40 == 0:
                print(f'  patch progress: {fetch_done}/{len(all_tasks)}')

    # ── PHASE 3: batch encode ─────────────────────────────────────────────────
    enc, etype = get_encoder()

    to_encode = []
    for s, y, d in all_tasks:
        slug   = f"{s.lower().replace(' ','_')}_{y}_{d}"
        s_path = f'/Users/jacindabietz/corn_nb/data/hls/{slug}.npy'
        e_path = f'/Users/jacindabietz/corn_nb/data/embeddings/{slug}.npy'
        if not os.path.exists(e_path) and os.path.exists(s_path):
            to_encode.append((slug, s_path, e_path))

    BATCH_SIZE = 16
    print(f'\nBatch encoding {len(to_encode)} patches (batch={BATCH_SIZE}, device={DEVICE}) ...')
    for i in range(0, len(to_encode), BATCH_SIZE):
        batch   = to_encode[i:i+BATCH_SIZE]
        patches = [np.load(sp) for _, sp, _ in batch]

        if etype == 'prithvi':
            tensors = []
            for patch in patches:
                p = patch[:6].copy()
                p = (p - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
                t = torch.from_numpy(p).float().unsqueeze(0)
                t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
                t = t.unsqueeze(2).repeat(1, 1, 3, 1, 1)    # 1 x 6 x 3 x 224 x 224
                tensors.append(t)
            batch_t = torch.cat(tensors, dim=0).to(DEVICE)   # B x 6 x 3 x 224 x 224
            with torch.no_grad():
                feats = enc.encoder.forward_features(batch_t)
            last = feats[-1]                                  # B x (1+n_patches) x 768
            embs = last[:, 1:, :].mean(dim=1).cpu().numpy()  # B x 768
        else:
            tensors = []
            for patch in patches:
                t = torch.from_numpy(patch).float().unsqueeze(0)
                t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
                tensors.append(t)
            batch_t = torch.cat(tensors, dim=0).to(DEVICE)
            with torch.no_grad():
                embs = enc(batch_t).cpu().numpy()

        for j, (_, _, e_path) in enumerate(batch):
            np.save(e_path, embs[j].flatten().astype(np.float32))

        print(f'  encoded {min(i + BATCH_SIZE, len(to_encode))}/{len(to_encode)}')

    # ── PHASE 4: assemble feature matrix (fast — everything already on disk) ──
    X_rows, y_vals, meta_rows = [], [], []

    for state in STATES:
        for year in YEARS:
            yield_val = nass_lookup.get((state, year), np.nan)
            if np.isnan(yield_val):
                print(f'  WARN: no NASS yield for {state} {year}, skipping')
                continue
            for date_key in DATES:
                slug   = f"{state.lower().replace(' ','_')}_{year}_{date_key}"
                s_path = f'/Users/jacindabietz/corn_nb/data/hls/{slug}.npy'
                e_path = f'/Users/jacindabietz/corn_nb/data/embeddings/{slug}.npy'

                if not os.path.exists(s_path) or not os.path.exists(e_path):
                    print(f'  SKIP: {slug} (missing patch or embedding)')
                    continue

                patch = np.load(s_path)
                emb   = np.load(e_path)

                cdl_mask = _load_cdl_mask(state, year)
                if cdl_mask is not None and cdl_mask.shape == patch.shape[1:]:
                    flat_mask = cdl_mask.ravel().astype(bool)
                    ndvi_vals = patch[6].ravel()[flat_mask]
                    evi_vals  = patch[7].ravel()[flat_mask]
                    lswi_vals = patch[8].ravel()[flat_mask]
                else:
                    ndvi_vals = patch[6].ravel()
                    evi_vals  = patch[7].ravel()
                    lswi_vals = patch[8].ravel()

                spectral = np.array([
                    float(ndvi_vals.mean()) if len(ndvi_vals) else float(patch[6].mean()),
                    float(ndvi_vals.std())  if len(ndvi_vals) else float(patch[6].std()),
                    float(evi_vals.mean())  if len(evi_vals)  else float(patch[7].mean()),
                    float(lswi_vals.mean()) if len(lswi_vals) else float(patch[8].mean()),
                ], dtype=np.float32)

                w_feats = weather_cache.get((state, year))
                if w_feats is None:
                    print(f'  SKIP: {slug} (missing weather)')
                    continue

                row = np.concatenate([emb, spectral, w_feats])
                X_rows.append(row)
                y_vals.append(yield_val)
                meta_rows.append({
                    'state'        : state,
                    'year'         : year,
                    'forecast_date': date_key,
                    'yield_bu_acre': yield_val,
                })

    X    = np.stack(X_rows, axis=0).astype(np.float32)
    y    = np.array(y_vals, dtype=np.float32)
    meta = pd.DataFrame(meta_rows)

    np.save(X_PATH,    X)
    np.save(Y_PATH,    y)
    meta.to_csv(META_PATH, index=False)
    print(f'\nSaved X -> {X_PATH}')
    print(f'Saved y -> {Y_PATH}')
    print(f'Saved metadata -> {META_PATH}')

print(f'\nX shape : {X.shape}')
print(f'y shape : {y.shape}')
print('\nmetadata sample:')
print(meta.head())

%pip install quantile-forest --quiet


In [ ]:
# Cell 9 -- QRF yield models: one per forecast date, leave-one-year-out CV
import os, json, numpy as np, pandas as pd, joblib
from quantile_forest import RandomForestQuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X    = np.load('/Users/jacindabietz/corn_nb/data/features/X.npy')
y    = np.load('/Users/jacindabietz/corn_nb/data/features/y.npy')
meta = pd.read_csv('/Users/jacindabietz/corn_nb/data/features/metadata.csv')

DATES     = ['aug1', 'sep1', 'oct1', 'eos']
YEARS     = sorted(meta['year'].unique())
QUANTILES = [0.1, 0.5, 0.9]
os.makedirs('/Users/jacindabietz/corn_nb/models',  exist_ok=True)
os.makedirs('/Users/jacindabietz/corn_nb/outputs', exist_ok=True)

QRF_PARAMS = dict(
    n_estimators     = 500,
    min_samples_leaf = 5,
    max_features     = 0.33,
    random_state     = 42,
    n_jobs           = -1,
)

cv_results   = {}
summary_rows = []

for date_key in DATES:
    mask   = meta['forecast_date'] == date_key
    X_d    = X[mask]
    y_d    = y[mask]
    meta_d = meta[mask].reset_index(drop=True)

    all_lows, all_preds, all_highs, all_true = [], [], [], []

    for held_year in YEARS:
        train_mask = meta_d['year'] != held_year
        test_mask  = meta_d['year'] == held_year
        if test_mask.sum() == 0:
            continue
        X_tr, y_tr = X_d[train_mask], y_d[train_mask]
        X_te, y_te = X_d[test_mask],  y_d[test_mask]

        qrf = RandomForestQuantileRegressor(**QRF_PARAMS)
        qrf.fit(X_tr, y_tr)

        qs = qrf.predict(X_te, quantiles=QUANTILES)  # (n_test, 3)
        all_lows.extend(qs[:, 0].tolist())
        all_preds.extend(qs[:, 1].tolist())
        all_highs.extend(qs[:, 2].tolist())
        all_true.extend(y_te.tolist())

    all_true  = np.array(all_true)
    all_preds = np.array(all_preds)
    all_lows  = np.array(all_lows)
    all_highs = np.array(all_highs)

    mae      = mean_absolute_error(all_true, all_preds)
    rmse     = mean_squared_error(all_true, all_preds) ** 0.5
    coverage = float(np.mean((all_lows <= all_true) & (all_true <= all_highs)))
    avg_width = float(np.mean(all_highs - all_lows))

    print(f'{date_key.upper():5s} CV  MAE={mae:.2f}  RMSE={rmse:.2f}  '
          f'90%-coverage={coverage:.1%}  avg-interval={avg_width:.1f} bu/acre')

    cv_results[date_key] = {
        'mae': round(mae, 3), 'rmse': round(rmse, 3),
        'coverage_90': round(coverage, 3), 'avg_interval': round(avg_width, 2),
    }
    summary_rows.append({
        'date': date_key, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
        '90% cover': f'{coverage:.1%}', 'avg width': f'{avg_width:.1f}',
    })

    # Final model trained on ALL data for 2025 prediction
    final = RandomForestQuantileRegressor(**QRF_PARAMS)
    final.fit(X_d, y_d)
    model_path = f'/Users/jacindabietz/corn_nb/models/yield_{date_key}.pkl'
    joblib.dump(final, model_path)
    print(f'  saved -> {model_path}')

with open('/Users/jacindabietz/corn_nb/outputs/cv_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)

print('\n-- Summary --')
print(pd.DataFrame(summary_rows).to_string(index=False))

maes = [cv_results[d]['mae'] for d in DATES]
if maes[-1] < maes[0]:
    print('\nOK: MAE narrows as the season progresses (as expected).')
else:
    print('\nWARN: MAE does not strictly narrow -- check feature quality.')


In [ ]:
# Cell 11 -- Full 2025 forecast: 5 states x 4 forecast dates
import os, json as _json, joblib, numpy as np, pandas as pd

X    = np.load('/Users/jacindabietz/corn_nb/data/features/X.npy')
y    = np.load('/Users/jacindabietz/corn_nb/data/features/y.npy')
meta = pd.read_csv('/Users/jacindabietz/corn_nb/data/features/metadata.csv').reset_index(drop=True)

STATES      = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
DATES       = ['aug1', 'sep1', 'oct1', 'eos']
DATE_LABELS = {'aug1': 'Aug 1', 'sep1': 'Sep 1', 'oct1': 'Oct 1', 'eos': 'EOS'}
QUANTILES   = [0.1, 0.5, 0.9]
os.makedirs('/Users/jacindabietz/corn_nb/outputs', exist_ok=True)

models = {}
for dk in DATES:
    mp = f'/Users/jacindabietz/corn_nb/models/yield_{dk}.pkl'
    if not os.path.exists(mp):
        raise FileNotFoundError(f'Model not found: {mp} -- run Cell 9 first.')
    models[dk] = joblib.load(mp)

FORECAST_YEAR = 2025
if not (meta['year'] == FORECAST_YEAR).any():
    print(f'WARNING: No {FORECAST_YEAR} rows found, falling back to 2024.')
    FORECAST_YEAR = 2024
else:
    print(f'Forecasting year: {FORECAST_YEAR}')

forecast_results = {}

for state in STATES:
    forecast_results[state] = {}
    for dk in DATES:
        row_mask = ((meta['state'] == state) & (meta['year'] == FORECAST_YEAR) &
                    (meta['forecast_date'] == dk))
        if row_mask.sum() == 0:
            print(f'  WARN: no {FORECAST_YEAR} feature row for {state} {dk}')
            continue

        feat_vec          = X[row_mask][0:1]
        low, pred, high   = models[dk].predict(feat_vec, quantiles=QUANTILES)[0]

        forecast_results[state][dk] = {
            'predicted_yield': round(float(pred), 2),
            'cone_low'       : round(float(low),  2),
            'cone_high'      : round(float(high), 2),
            'cone_width'     : round(float(high - low), 2),
        }

with open('/Users/jacindabietz/corn_nb/outputs/forecast_results.json', 'w') as f:
    _json.dump(forecast_results, f, indent=2)
print('Saved -> /Users/jacindabietz/corn_nb/outputs/forecast_results.json\n')

rows = []
for state in STATES:
    row = {'State': state}
    for dk in DATES:
        r = forecast_results.get(state, {}).get(dk, {})
        row[DATE_LABELS[dk]] = (f"{r['predicted_yield']:.1f} ({r['cone_low']:.1f}-{r['cone_high']:.1f})"
                                if r else 'N/A')
    rows.append(row)

print(pd.DataFrame(rows).set_index('State').to_string())


In [ ]:
# Cell 12 -- Cone of uncertainty plots: 5 states, 2025 forecast
import json, numpy as np, matplotlib.pyplot as plt

with open('/Users/jacindabietz/corn_nb/outputs/forecast_results.json') as f:
    results = json.load(f)

STATES      = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
DATES       = ['aug1', 'sep1', 'oct1', 'eos']
DATE_LABELS = ['Aug 1', 'Sep 1', 'Oct 1', 'EOS']
COLORS      = ['#2ca02c', '#1f77b4', '#ff7f0e', '#9467bd', '#d62728']
x_pos       = np.arange(len(DATES))

fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
fig.suptitle('2025 Corn Yield Forecast — Cone of Uncertainty (QRF 10th–90th percentile)',
             fontsize=14, y=1.01)

for ax, state, color in zip(axes, STATES, COLORS):
    sr    = results.get(state, {})
    preds = [sr.get(d, {}).get('predicted_yield', np.nan) for d in DATES]
    lows  = [sr.get(d, {}).get('cone_low',        np.nan) for d in DATES]
    highs = [sr.get(d, {}).get('cone_high',       np.nan) for d in DATES]

    ax.fill_between(x_pos, lows, highs, alpha=0.25, color=color)
    ax.plot(x_pos, preds, 'o-', color=color, linewidth=2)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(DATE_LABELS, rotation=30, ha='right', fontsize=9)
    ax.set_xlabel('Forecast Date', fontsize=9)
    ax.set_ylabel('Yield (bu/acre)', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_title(state, fontsize=11)

plt.tight_layout()
plt.savefig('/Users/jacindabietz/corn_nb/outputs/cone_plot_qrf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> /Users/jacindabietz/corn_nb/outputs/cone_plot_qrf.png')


In [ ]:
# Cell 13 -- Choropleth: predicted 2025 EOS corn yield across 5 states
import os, json, zipfile, requests
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt

with open('/Users/jacindabietz/corn_nb/outputs/forecast_results.json') as f:
    results = json.load(f)

TARGET_STATES = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
STATE_ABBREVS = {'Iowa': 'IA', 'Colorado': 'CO', 'Wisconsin': 'WI',
                 'Missouri': 'MO', 'Nebraska': 'NE'}

eos_yields = {
    s: results[s]['eos']['predicted_yield']
    for s in TARGET_STATES
    if 'eos' in results.get(s, {})
}

SHP_ZIP = '/Users/jacindabietz/corn_nb/data/cb_2022_us_state_5m.zip'
SHP_DIR = '/Users/jacindabietz/corn_nb/data/cb_2022_us_state_5m'

if not os.path.exists(SHP_ZIP):
    url = ('https://www2.census.gov/geo/tiger/GENZ2022/shp/'
           'cb_2022_us_state_5m.zip')
    print(f'Downloading shapefile from Census TIGER ...')
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(SHP_ZIP, 'wb') as fh:
        fh.write(r.content)
    print('Downloaded.')

if not os.path.exists(SHP_DIR):
    with zipfile.ZipFile(SHP_ZIP, 'r') as zf:
        zf.extractall(SHP_DIR)
    print(f'Extracted -> {SHP_DIR}')

shp_file = [f for f in os.listdir(SHP_DIR) if f.endswith('.shp')][0]
gdf      = gpd.read_file(os.path.join(SHP_DIR, shp_file))
gdf_sub  = gdf[gdf['NAME'].isin(TARGET_STATES)].copy()
gdf_sub['predicted_yield'] = gdf_sub['NAME'].map(eos_yields)
gdf_sub['abbrev']          = gdf_sub['NAME'].map(STATE_ABBREVS)

fig, ax = plt.subplots(figsize=(12, 8))
gdf_sub.plot(
    column    = 'predicted_yield',
    cmap      = 'YlGn',
    legend    = True,
    ax        = ax,
    edgecolor = 'black',
    linewidth = 0.8,
    legend_kwds = {'label': 'Predicted Yield (bu/acre)',
                   'orientation': 'vertical'},
)
for _, row in gdf_sub.iterrows():
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    ax.annotate(f"{row['abbrev']}\n{row['predicted_yield']:.1f}",
                xy=(cx, cy), ha='center', va='center',
                fontsize=10, fontweight='bold', color='black')

ax.set_title('Predicted 2025 EOS Corn Yield (bu/acre)', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('/Users/jacindabietz/corn_nb/outputs/choropleth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> /Users/jacindabietz/corn_nb/outputs/choropleth.png')


In [ ]:
# Cell 13b -- Interactive folium choropleth: 2025 forecast across 4 dates
import os, json, zipfile, requests
import numpy as np, pandas as pd, geopandas as gpd
import folium
from folium.plugins import FeatureGroupSubGroup

with open('/Users/jacindabietz/corn_nb/outputs/forecast_results.json') as f:
    results = json.load(f)

TARGET_STATES = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
DATES         = ['aug1', 'sep1', 'oct1', 'eos']
DATE_LABELS   = {'aug1': 'Aug 1', 'sep1': 'Sep 1', 'oct1': 'Oct 1', 'eos': 'End of Season'}

# ── load shapefile (already cached from Cell 13) ──────────────────────────
SHP_ZIP = '/Users/jacindabietz/corn_nb/data/cb_2022_us_state_5m.zip'
SHP_DIR = '/Users/jacindabietz/corn_nb/data/cb_2022_us_state_5m'

if not os.path.exists(SHP_ZIP):
    url = 'https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_5m.zip'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(SHP_ZIP, 'wb') as fh:
        fh.write(r.content)

if not os.path.exists(SHP_DIR):
    with zipfile.ZipFile(SHP_ZIP, 'r') as zf:
        zf.extractall(SHP_DIR)

shp_file = [f for f in os.listdir(SHP_DIR) if f.endswith('.shp')][0]
gdf_all  = gpd.read_file(os.path.join(SHP_DIR, shp_file))
gdf      = gdf_all[gdf_all['NAME'].isin(TARGET_STATES)].copy()

# ── attach yield predictions for each forecast date ───────────────────────
for d in DATES:
    gdf[d] = gdf['NAME'].map(
        {s: results.get(s, {}).get(d, {}).get('predicted_yield', float('nan'))
         for s in TARGET_STATES}
    )

# ── build map ─────────────────────────────────────────────────────────────
m = folium.Map(location=[41.5, -97.0], zoom_start=5, tiles='CartoDB positron')

all_yields = [gdf[d].dropna().values for d in DATES]
vmin = float(np.nanmin([v.min() for v in all_yields if len(v)]))
vmax = float(np.nanmax([v.max() for v in all_yields if len(v)]))

colormap = folium.LinearColormap(
    colors=['#ffffcc', '#78c679', '#238443'],
    vmin=vmin, vmax=vmax,
    caption='Predicted Corn Yield (bu/acre)'
)
colormap.add_to(m)

for d in DATES:
    label = DATE_LABELS[d]
    fg    = folium.FeatureGroup(name=label, show=(d == 'eos'))

    geojson_data = json.loads(gdf[['NAME', d, 'geometry']].to_json())

    folium.GeoJson(
        geojson_data,
        style_function=lambda feat, col=d: {
            'fillColor'  : colormap(feat['properties'][col])
                           if feat['properties'][col] is not None
                           else '#cccccc',
            'color'      : '#333333',
            'weight'     : 1.2,
            'fillOpacity': 0.75,
        },
        tooltip=folium.GeoJsonTooltip(
            fields   = ['NAME', d],
            aliases  = ['State', f'{label} yield (bu/acre)'],
            localize = True,
        ),
    ).add_to(fg)

    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

out_html = '/Users/jacindabietz/corn_nb/outputs/choropleth_interactive.html'
os.makedirs(os.path.dirname(out_html), exist_ok=True)
m.save(out_html)
print(f'Saved -> {out_html}')
m


In [ ]:
# Cell 14 -- Feature importance for EOS yield model
import os, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt

model = joblib.load('/Users/jacindabietz/corn_nb/models/yield_eos.pkl')

X    = np.load('/Users/jacindabietz/corn_nb/data/features/X.npy')
meta = pd.read_csv('/Users/jacindabietz/corn_nb/data/features/metadata.csv').reset_index(drop=True)

mask  = meta['forecast_date'] == 'eos'
X_eos = X[mask]

n_emb      = X_eos.shape[1] - 10
feat_names = (
    [f'emb_{i}' for i in range(n_emb)]
    + ['mean_ndvi', 'std_ndvi', 'mean_evi', 'mean_lswi']
    + ['mean_tmax', 'mean_tmin', 'total_precip',
       'max_tmax',  'min_tmin',  'precip_std']
)

print(f'X_eos shape  : {X_eos.shape}')
print(f'Feature count: {len(feat_names)}')

importances = model.feature_importances_
top20_idx   = np.argsort(importances)[::-1][:20]

print('
Top 5 features by MDI importance:')
for rank, idx in enumerate(top20_idx[:5], 1):
    print(f'  {rank}. {feat_names[idx]:20s}  importance={importances[idx]:.4f}')

# Bar chart (top 20)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    [feat_names[i] for i in top20_idx[::-1]],
    importances[top20_idx[::-1]],
    color='steelblue',
)
ax.set_xlabel('Mean Decrease Impurity')
ax.set_title('Feature Importance -- EOS Yield Model', fontsize=12)
plt.tight_layout()
plt.savefig('/Users/jacindabietz/corn_nb/outputs/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> /Users/jacindabietz/corn_nb/outputs/shap_bar.png')

# Embedding vs. weather/veg breakdown
groups = {
    'Prithvi embeddings': [i for i, n in enumerate(feat_names) if n.startswith('emb_')],
    'Vegetation indices': [i for i, n in enumerate(feat_names) if n in ('mean_ndvi','std_ndvi','mean_evi','mean_lswi')],
    'Weather':            [i for i, n in enumerate(feat_names) if 'tmax' in n or 'tmin' in n or 'precip' in n],
}
group_imp = {k: importances[v].sum() for k, v in groups.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(group_imp.keys(), group_imp.values(), color=['steelblue','forestgreen','coral'])
ax.set_ylabel('Total MDI importance')
ax.set_title('Feature group importance -- EOS model', fontsize=12)
plt.tight_layout()
plt.savefig('/Users/jacindabietz/corn_nb/outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> /Users/jacindabietz/corn_nb/outputs/shap_beeswarm.png')


In [ ]:
# Cell 15 -- Export and final summary
import os, json
import numpy as np, pandas as pd

STATES      = ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
DATES       = ['aug1', 'sep1', 'oct1', 'eos']
DATE_LABELS = {'aug1': 'Aug 1', 'sep1': 'Sep 1', 'oct1': 'Oct 1', 'eos': 'EOS'}

# 1. file checklist
expected = [
    '/Users/jacindabietz/corn_nb/outputs/forecast_results.json',
    '/Users/jacindabietz/corn_nb/outputs/cone_plot.png',
    '/Users/jacindabietz/corn_nb/outputs/choropleth.png',
    '/Users/jacindabietz/corn_nb/outputs/shap_bar.png',
    '/Users/jacindabietz/corn_nb/outputs/shap_beeswarm.png',
]
print('Output file checklist:')
for path in expected:
    mark = '[OK]' if os.path.exists(path) else '[MISSING]'
    print(f'  {os.path.basename(path):<35s}  {mark}')

# 2. flat CSV export
with open('/Users/jacindabietz/corn_nb/outputs/forecast_results.json') as f:
    results = json.load(f)

rows = []
for state in STATES:
    for dk in DATES:
        r = results.get(state, {}).get(dk, {})
        rows.append({
            'state'          : state,
            'forecast_date'  : DATE_LABELS[dk],
            'predicted_yield': r.get('predicted_yield', np.nan),
            'cone_low'       : r.get('cone_low',        np.nan),
            'cone_high'      : r.get('cone_high',       np.nan),
            'cone_width'     : r.get('cone_width',      np.nan),
        })

summary_df = pd.DataFrame(rows)
csv_path   = '/Users/jacindabietz/corn_nb/outputs/forecast_summary.csv'
summary_df.to_csv(csv_path, index=False)
print(f'\nSaved -> {csv_path}')
print(summary_df.to_string(index=False))

# 3. human-readable per-state summary
print('\n-- Per-state forecast narrative --')
for state in STATES:
    sr     = results.get(state, {})
    preds  = [sr.get(d, {}).get('predicted_yield', float('nan')) for d in DATES]
    widths = [sr.get(d, {}).get('cone_width',       float('nan')) for d in DATES]
    parts  = ' -> '.join(
        f'{DATE_LABELS[d]}={p:.1f}'
        for d, p in zip(DATES, preds)
        if not np.isnan(p)
    )
    w_valid    = [w for w in widths if not np.isnan(w)]
    width_note = (f'  (cone width: {w_valid[0]:.1f} -> {w_valid[-1]:.1f} bu/ac)'
                  if len(w_valid) >= 2 else '')
    print(f'  {state}: {parts} bu/ac{width_note}')

# 4. CV performance recap
cv_path = '/Users/jacindabietz/corn_nb/outputs/cv_results.json'
if os.path.exists(cv_path):
    with open(cv_path) as f:
        cv = json.load(f)
    best_date = min(cv, key=lambda d: cv[d]['mae'])
    print('\n-- Model performance recap --')
    for dk in DATES:
        print(f'  {DATE_LABELS[dk]:6s}: MAE={cv[dk]["mae"]:.2f}  '
              f'RMSE={cv[dk]["rmse"]:.2f} bu/acre')
    print(f'\n  Best forecast date by MAE: {DATE_LABELS[best_date]} '
          f'at {cv[best_date]["mae"]:.2f} bu/acre')
else:
    print('\nWARN: cv_results.json not found -- run Cell 9 first.')

# 5. write README
readme = (
    '# Corn-for-Grain Yield Forecasting Pipeline\n\n'
    'This notebook builds an end-to-end corn yield forecasting system for five\n'
    'US states (Iowa, Colorado, Wisconsin, Missouri, Nebraska) across growing\n'
    'seasons 2005-2024. Satellite imagery from NASA HLS (HLSS30, Sentinel-2\n'
    'harmonized, 30 m) is fetched via the earthaccess library and encoded with\n'
    'the Prithvi-100M geospatial foundation model, combined with NASA POWER\n'
    'weather aggregates, USDA NASS reference yields, and USDA CDL corn masks\n'
    'to train XGBoost regressors. Forecasts are issued at four points in the\n'
    'season (Aug 1, Sep 1, Oct 1, EOS) for target year 2025 (falls back to\n'
    '2024 if 2025 satellite data is unavailable) and visualised as a cone of\n'
    'uncertainty via analog year matching.\n\n'
    '## Data sources\n'
    '- **USDA NASS** -- official corn yield records (bu/acre) via NASS\n'
    '  QuickStats API (2005-2024)\n'
    '- **NASA HLS HLSS30** -- Sentinel-2 harmonized surface reflectance\n'
    '  (30 m, bands B02/B03/B04/B8A/B11/B12) via earthaccess; Planetary\n'
    '  Computer Sentinel-2 L2A is the fallback\n'
    '- **USDA CDL** -- Cropland Data Layer rasters from USDA CropScape WCS;\n'
    '  used to mask non-corn pixels before computing spectral aggregates\n'
    '- **NAIP** -- 4-band (R/G/B/NIR) aerial imagery via Planetary Computer STAC\n'
    '- **Prithvi-100M** (ibm-nasa-geospatial/Prithvi-100M) -- geospatial\n'
    '  foundation model used as the satellite image encoder (ResNet-50 fallback\n'
    '  if unavailable)\n'
    '- **NASA POWER** -- daily T2M_MAX/T2M_MIN/PRECTOTCORR for growing season\n'
    '  (April-October) aggregated to 6 scalar features; Open-Meteo is the fallback\n\n'
    '## Output files\n'
    '| File | Description |\n'
    '|------|-------------|\n'
    '| `forecast_results.json` | Nested dict: state -> forecast_date -> predicted yield + cone |\n'
    '| `forecast_summary.csv`  | Flat CSV of forecast_results |\n'
    '| `cone_plot.png`         | 5-panel cone of uncertainty chart (2025 forecast) |\n'
    '| `choropleth.png`        | US state choropleth of EOS predicted yield |\n'
    '| `shap_bar.png`          | SHAP mean absolute importance bar chart (EOS model) |\n'
    '| `shap_beeswarm.png`     | SHAP beeswarm showing feature effect directions |\n\n'
    '## How to re-run\n'
    '1. Ensure `/Users/jacindabietz/corn_nb/.env` contains:\n'
    '   `NASS_API_KEY`, `SH_CLIENT_ID`, `SH_CLIENT_SECRET`,\n'
    '   `ANTHROPIC_API_KEY`, `EARTHDATA_USERNAME`, `EARTHDATA_PASSWORD`\n'
    '2. Run cells 1-15 in order from a clean kernel.\n'
    '3. Cached `.npy` and `.csv` files under `/Users/jacindabietz/corn_nb/data/` allow individual\n'
    '   cells to be re-run without re-fetching remote data.\n'
    '4. GPU (CUDA) is used automatically for the Prithvi encoder and XGBoost\n'
    '   training if available (RTX A4000 recommended).\n'
)
readme_path = '/Users/jacindabietz/corn_nb/outputs/README.md'
with open(readme_path, 'w') as fh:
    fh.write(readme)
print(f'\nWrote -> {readme_path}')
print('All done.')